In [1]:
import gc
import os
import sys
import glob
import re

import numpy as np
import optuna
import pandas as pd
import torch

from tqdm import tqdm

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
%load_ext autoreload
%autoreload 2

from drGAT import drGAT

In [3]:
train_data = pd.read_csv("../GDSC1_data/train.csv")
val_data = pd.read_csv("../GDSC1_data/val.csv")
test_data = pd.read_csv("../GDSC1_data/test.csv")

In [4]:
train_data.head()

,Drug,Cell
0,PI-103,KU812
1,Tipifarnib,ETK-1
2,Apitolisib,NCI-H64
3,Docetaxel,UMC-11
4,Bleomycin,KYSE-220


In [5]:
idxs = np.load("../GDSC1_data/idxs.npy", allow_pickle=True)
converter = {idxs[1, i]: int(idxs[0, i]) for i in range(idxs.shape[1])}

In [6]:
def get_idx(X):
    X["Drug"] = [converter[(i)] for i in X["Drug"]]
    X["Cell"] = [converter[(i)] for i in X["Cell"]]
    return X

In [7]:
train_data = get_idx(train_data)
val_data = get_idx(val_data)
test_data = get_idx(test_data)

In [8]:
train_drug = train_data["Drug"].values
train_cell = train_data["Cell"].values
val_drug = val_data["Drug"].values
val_cell = val_data["Cell"].values

In [9]:
train_labels = np.load("../GDSC1_data/train_labels.npy")
val_labels = np.load("../GDSC1_data/val_labels.npy")

train_labels = torch.tensor(train_labels).float()
val_labels = torch.tensor(val_labels).float()

In [10]:
def load_and_combine_chunks(pattern, axis=0):
    chunk_files = sorted(
        glob.glob(pattern), key=lambda x: int(x.split("_")[-1].split(".")[0])
    )

    chunks = [np.load(f) for f in chunk_files]
    return np.concatenate(chunks, axis=axis)


edge_index = load_and_combine_chunks("../nci_data/edge_idxs/*.npy", axis=1)
edge_attr = load_and_combine_chunks("../nci_data/edge_attrs/*.npy", axis=0)
edge_index = torch.tensor(edge_index).int()
edge_index = edge_index.type(torch.int64)
edge_attr = torch.tensor(edge_attr).float()

## Get feature matrix

In [11]:
cell = pd.read_csv("../nci_data/cell_sim.csv", index_col=0)


# How to read
def natural_sort_key(s):
    return [int(c) if c.isdigit() else c for c in re.split(r"(\d+)", s)]


file_paths = glob.glob("../nci_data/drug_sim/drug_sim_part_*.parquet")
sorted_file_paths = sorted(file_paths, key=natural_sort_key)

drug = pd.concat([pd.read_parquet(file) for file in tqdm(sorted_file_paths)])


file_paths = glob.glob("../nci_data/gene_sim/gene_sim_part_*.parquet")
sorted_file_paths = sorted(file_paths, key=natural_sort_key)

gene = pd.concat([pd.read_parquet(file) for file in tqdm(sorted_file_paths)])

100%|██████████| 25/25 [00:02<00:00, 11.16it/s]


In [12]:
drug = torch.tensor(drug.values).float()
cell = torch.tensor(cell.values).float()
gene = torch.tensor(gene.values).float()

# Create the dataset

In [13]:
data = [
    drug,
    cell,
    gene,
    edge_index,
    edge_attr,
    train_drug,
    train_cell,
    val_drug,
    val_cell,
    train_labels,
    val_labels,
]

In [16]:
def objective(trial):
    params = {
        "n_drug": drug.shape[0],
        "n_cell": cell.shape[0],
        "n_gene": gene.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        "epochs": trial.suggest_int("epochs", 10, 200, step=50),
        "heads": trial.suggest_categorical("heads", [1, 2, 3]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer", ["GAT", "GATv2", "Transformer"]
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        _, _, _, best_metrics, early_stopping_epoch = drGAT.train(
            data, params=params, device=device, verbose=False
        )
        print("#####")
        print(best_metrics)
        print("#####")

        early_stop_threshold = trial.suggest_float("early_stop_threshold", 0.3, 0.7)
        if (
            early_stopping_epoch is not None
            and early_stopping_epoch < params["epochs"] * early_stop_threshold
        ):
            raise optuna.TrialPruned("Early stopping occurred too early")

        trial.set_user_attr("early_stopping_epoch", early_stopping_epoch)
        return best_metrics

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")
            
            # メモリ解放処理
            with torch.cuda.device('cuda'):
                torch.cuda.empty_cache()
            gc.collect()
            
            # Pruning通知
            raise optuna.TrialPruned(f"OOM at trial {trial.number}")
        
        else:
            raise e

In [18]:
name = "NCI_GAT"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}.sqlite3".format(name),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100)

[I 2025-03-22 16:54:51,258] A new study created in RDB with name: NCI_GAT
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/distributions.py:700: UserWarning: The distribution is specified by [10, 200] and step=50, but the range is not divisible by `step`. It will be replaced by [10, 160].
  warnings.warn(


Using:  cuda


  0%|          | 0/10 [00:00<?, ?it/s]
[I 2025-03-22 16:54:52,078] Trial 0 pruned. OOM at trial 0


Pruned trial 0: CUDA OOM
Using:  cuda


  0%|          | 0/60 [00:00<?, ?it/s]

...

Best model found at epoch 10
#####
[0.5221187427240978, 0.5321668803738115, 0.5353726955972575, 0.6433845886543307]
#####
Using:  cuda


100%|██████████| 10/10 [00:06<00:00,  1.61it/s]
[I 2025-03-22 16:59:15,389] Trial 21 finished with values: [0.5011641443538999, 0.5120182913903644, 0.5148901516640216, 0.16226783968719455] and parameters: {'dropout1': 0.3, 'dropout2': 0.5, 'hidden1': 1028, 'hidden2': 256, 'hidden3': 128, 'epochs': 10, 'heads': 1, 'activation': 'swish', 'optimizer': 'AdamW', 'lr': 5.91942772571701e-05, 'weight_decay': 4.386066453071743e-06, 'scheduler': None, 'gnn_layer': 'Transformer', 'amsgrad': False, 'early_stop_threshold': 0.3380671542172739}. 
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/distributions.py:700: UserWarning: The distribution is specified by [10, 200] and step=50, but the range is not divisible by `step`. It will be replaced by [10, 160].
  warnings.warn(


Best model found at epoch 10
#####
[0.5011641443538999, 0.5120182913903644, 0.5148901516640216, 0.16226783968719455]
#####
Using:  cuda


 60%|██████    | 6/10 [00:07<00:05,  1.29s/it]
[W 2025-03-22 16:59:23,560] Trial 22 failed with parameters: {'dropout1': 0.3, 'dropout2': 0.5, 'hidden1': 1028, 'hidden2': 256, 'hidden3': 128, 'epochs': 10, 'heads': 2, 'activation': 'swish', 'optimizer': 'AdamW', 'lr': 2.1287801988965917e-05, 'weight_decay': 6.955209922577344e-06, 'scheduler': 'Cosine', 'gnn_layer': 'Transformer', 'T_max': 26, 'amsgrad': False} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/study/_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_3536071/3415701248.py", line 71, in objective
    _, _, _, best_metrics, early_stopping_epoch = drGAT.train(
  File "/spin1/home/linux/inouey2/drGAT/drGAT/drGAT.py", line 238, in train
    train_attention = train_one_epoch(
  File "/spin1/home/linux/inouey2/drGAT/drGAT/drGAT.py", line 378, in train_one_epoch
    scaler.

KeyboardInterrupt: 

## Eval

In [ ]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [22]:
# prob, res, test_attention = drGAT.eval(model, test)